# Step 7: Statistical Analysis & Hypothesis Testing
Rigorous testing:
1. Independent Two-Sample T-Test (Delivery Status vs Review Score)
2. One-Way ANOVA (Product Category vs Order Value)
3. Chi-Square Test of Independence (Payment Method vs Order Status)


In [1]:
import pandas as pd
import sqlite3
from scipy import stats
import os

import sys
import os

def resolve_path(rel_path):
    curr = os.path.abspath(os.getcwd())
    while curr and os.path.dirname(curr) != curr:
        candidate = os.path.join(curr, rel_path)
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
        curr = os.path.dirname(curr)
    return os.path.abspath(rel_path)

db_path = resolve_path("data/cleaned/ecommerce.db")
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA temp_store = MEMORY;")
orders = pd.read_sql("SELECT * FROM orders_features", conn)
reviews = pd.read_sql("SELECT * FROM order_reviews", conn)

df = orders.merge(reviews, on='order_id')
delayed = df[df['is_delayed'] == 1]['review_score'].dropna()
ontime = df[df['is_delayed'] == 0]['review_score'].dropna()

t_stat, p_val = stats.ttest_ind(delayed, ontime, equal_var=False)
print(f"Independent T-Test Results: T-statistic={t_stat:.4f}, p-value={p_val:.4e}", flush=True)
if p_val < 0.05:
    print("Conclusion: Reject H0. Delayed orders receive significantly lower review scores.", flush=True)
else:
    print("Conclusion: Fail to reject H0.", flush=True)
conn.close()



Independent T-Test Results: T-statistic=-85.2324, p-value=0.0000e+00
Conclusion: Reject H0. Delayed orders receive significantly lower review scores.
